# EDA - STAGING photos (198F, 33F, 48F only)

Exploratory Data Analysis for the digit detection pipeline using **STAGING_FOLDER** as input and only photo IDs **198F**, **33F**, and **48F** (same as digit prediction service).

In [ ]:
# Add workspace root to path so base_modules can be imported
import sys
from pathlib import Path

ETL_DIR = Path('.').resolve()
WEBAPP_DIR = ETL_DIR.parent
WORKSPACE_ROOT = WEBAPP_DIR.parent
sys.path.insert(0, str(WORKSPACE_ROOT))

import cv2
import numpy as np
import matplotlib.pyplot as plt

from base_modules import (
    STAGING_FOLDER,
    PHOTO_ID_PARAMS,
    get_params_for_photo_id,
    DEFAULT_CROP,
    adjust_brightness_contrast,
    apply_gaussian_blur,
    preprocess_for_edge_detection,
)

# Photo IDs used by digit prediction service (only these are considered)
TARGET_PHOTO_IDS = {'198F', '33F', '48F'}

def extract_photo_id(filename):
    """Extract photo ID from filename (3rd field when split by '-')."""
    stem = Path(filename).stem
    parts = stem.split('-')
    return parts[2] if len(parts) >= 3 else ''

print(f"Staging folder: {STAGING_FOLDER}")
print(f"Staging exists: {STAGING_FOLDER.exists()}")
print(f"Target photo IDs: {TARGET_PHOTO_IDS}")

## 1. List images in STAGING (198F, 33F, 48F only)

In [ ]:
# List image files in STAGING and filter by target photo IDs
extensions = ('*.bmp', '*.png', '*.jpg', '*.jpeg')
all_files = []
for ext in extensions:
    all_files.extend(STAGING_FOLDER.glob(ext))

bmp_files = [f for f in all_files if extract_photo_id(f.name) in TARGET_PHOTO_IDS]
bmp_files.sort(key=lambda p: p.name)

print(f"Found {len(bmp_files)} images in STAGING with photo ID in {TARGET_PHOTO_IDS}")
if bmp_files:
    print("\nFirst 10 files:")
    for f in bmp_files[:10]:
        print(f"  - {f.name} (ID: {extract_photo_id(f.name)})")
else:
    print("No matching images. Add photos to STAGING with filenames containing -198F-, -33F-, or -48F-.")

## 2. Load sample image

In [ ]:
# Select a sample (change index to analyze different images)
sample_index = 0
if not bmp_files:
    raise FileNotFoundError("No images found in STAGING for 198F/33F/48F. Add photos first.")
sample_path = bmp_files[sample_index]
sample_name = sample_path.name
photo_id = extract_photo_id(sample_name)
image = cv2.imread(str(sample_path))
if image is None:
    raise ValueError(f"Could not read image: {sample_path}")
print(f"Sample: {sample_name}")
print(f"Photo ID: {photo_id}")
print(f"Shape: {image.shape}")

In [ ]:
# Rotate 90° clockwise (same as digit prediction service)
rotated = cv2.rotate(image, cv2.ROTATE_90_CLOCKWISE)

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
ax[0].set_title('Original')
ax[0].axis('off')
ax[1].imshow(cv2.cvtColor(rotated, cv2.COLOR_BGR2RGB))
ax[1].set_title('Rotated 90° CW')
ax[1].axis('off')
plt.tight_layout()
plt.show()

## 3. Crop region of interest

In [ ]:
x1, y1 = DEFAULT_CROP['x1'], DEFAULT_CROP['y1']
x2, y2 = DEFAULT_CROP['x2'], DEFAULT_CROP['y2']
cropped = rotated[y1:y2, x1:x2]

fig, ax = plt.subplots(1, 1, figsize=(8, 6))
ax.imshow(cv2.cvtColor(cropped, cv2.COLOR_BGR2RGB))
ax.set_title(f'Cropped region (ID: {photo_id})')
ax.axis('off')
plt.show()

## 4. Preprocessing pipeline (edge detection)

In [ ]:
params = get_params_for_photo_id(photo_id)
print(f"Parameters for {photo_id}: {params}")

gray = cv2.cvtColor(cropped, cv2.COLOR_BGR2GRAY)
edges = preprocess_for_edge_detection(
    gray,
    brightness=params['brightness'],
    contrast=params['contrast'],
    canny_low=params['canny_low'],
    canny_high=params['canny_high'],
    gaussian_blur=params.get('gaussian_blur', 5),
    negative=params.get('negative', False),
)

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].imshow(gray, cmap='gray')
ax[0].set_title('Grayscale crop')
ax[0].axis('off')
ax[1].imshow(edges, cmap='gray')
ax[1].set_title('Edge detection (preprocessing output)')
ax[1].axis('off')
plt.tight_layout()
plt.show()

## 5. Multiple samples (198F, 33F, 48F)

In [ ]:
# Process up to 6 samples (2 per row)

n_show = min(6, len(bmp_files))
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for idx in range(n_show):
    path = bmp_files[idx]
    pid = extract_photo_id(path.name)
    img = cv2.imread(str(path))
    if img is None:
        continue
    rot = cv2.rotate(img, cv2.ROTATE_90_CLOCKWISE)
    x1, y1 = DEFAULT_CROP['x1'], DEFAULT_CROP['y1']
    x2, y2 = DEFAULT_CROP['x2'], DEFAULT_CROP['y2']
    crop = rot[y1:y2, x1:x2]
    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    prm = get_params_for_photo_id(pid)
    edg = preprocess_for_edge_detection(
        gray,
        brightness=prm['brightness'],
        contrast=prm['contrast'],
        canny_low=prm['canny_low'],
        canny_high=prm['canny_high'],
        gaussian_blur=prm.get('gaussian_blur', 5),
        negative=prm.get('negative', False),
    )
    axes[idx].imshow(edg, cmap='gray')
    axes[idx].set_title(f'{path.name[:30]}... (ID: {pid})')
    axes[idx].axis('off')

for j in range(n_show, 6):
    axes[j].axis('off')
plt.tight_layout()
plt.show()

## 6. Photo ID parameters (198F, 33F, 48F)

In [ ]:
import pandas as pd
subset = {k: v for k, v in PHOTO_ID_PARAMS.items() if k in TARGET_PHOTO_IDS or k == 'default'}
df = pd.DataFrame(subset).T
print("Parameters for target photo IDs:")
display(df)